# Notebook overview

This notebook demonstrates how to ingest meeting notes into LlamaIndex,
apply HuggingFace embeddings, and build a query engine for retrieval.
It also shows how to configure Langfuse tracing and enable OpenInference
instrumentation for LlamaIndex.

Purpose
- Ingest meeting notes from `meeting_notes/` and split them into searchable
  chunks.
- Create embeddings using `sentence-transformers/all-MiniLM-L6-v2`.
- Build a vector index and run natural-language queries over the data.
- Configure optional Langfuse monitoring and instrumentation.

What the notebook covers
1. Environment setup and secret loading.
2. Langfuse client initialization.
3. Installation and setup of LlamaIndex instrumentation.
4. Document loading, chunking, embedding, and index creation.
5. Querying the index and inspecting the LLM stack.

Run instructions
- Install required packages: `python -m pip install -r requirements.txt`
- Create a `.env` file with required API keys and endpoints.
- Run the cells in order from top to bottom.

Security note
- Avoid putting secrets directly in notebook cells. Use environment variables or
  secure vaults instead.

# Set the OpenAI-compatible API key
This cell loads the API key from the environment and assigns it for use by
LlamaIndex/OpenAI-style clients.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OMNIROUTER_API_KEY")

# Load environment variables and configure Langfuse settings
This cell loads `.env` values and sets Langfuse-related environment variables
for the tracing client.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
secret_key = os.getenv("LANGFUSE_SECRET_KEY")
base_url = os.getenv("LANGFUSE_BASE_URL")

os.environ.setdefault("LANGFUSE_PUBLIC_KEY", public_key)
os.environ.setdefault("LANGFUSE_SECRET_KEY", secret_key)
os.environ.setdefault("LANGFUSE_BASE_URL", base_url)

# Initialize the Langfuse client
This cell creates a Langfuse client and verifies authentication so tracing
works correctly before proceeding.

In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

# Install the LlamaIndex instrumentation package
This cell installs the package required to instrument LlamaIndex calls for
monitoring and tracing.

In [ ]:
pip install openinference-instrumentation-llama-index

# Enable OpenInference instrumentation for LlamaIndex
This cell instruments LlamaIndex so OpenInference can capture model and query
metrics during index construction and querying.

In [ ]:
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor

# Initialize LlamaIndex instrumentation
LlamaIndexInstrumentor().instrument()

# Imports and setup
Imports the LlamaIndex components, loads environment variables, and prepares
document loading as well as the embedding and LLM configuration.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.llms.openai_like import OpenAILike



import os
from dotenv import load_dotenv

load_dotenv()

# Load documents
documents = SimpleDirectoryReader(input_dir="meeting_notes").load_data()
print(f"Loaded {len(documents)} documents")

# Split into Chunks
splitter = SentenceSplitter(chunk_size=300)
nodes = splitter.get_nodes_from_documents(documents)

# Setup LLM 
model = os.getenv("LLM_MODEL", "gpt-4o-mini")
Settings.llm = OpenAILike(
    model=model,
    api_key=os.getenv("OMNIROUTER_API_KEY"),
    api_base=os.getenv("OMNIROUTER_BASE_URL"),
    streaming=False,
    is_chat_model=True
)
embeddings = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

Settings.embed_model = embeddings

# Create a VectorStoreIndex from the nodes
index = VectorStoreIndex(nodes=nodes)
print("VectorStoreIndex created from document nodes.")

# Create a query engine
query_engine = index.as_query_engine()
print("Query engine initialized. You can now ask questions!")

# Inspect LLM
Quick check to print the LLM object used by the response synthesizer.

In [ ]:
print(query_engine._response_synthesizer._llm)

# Run a sample query
Executes a sample query against the `query_engine` and prints the result.

In [ ]:
response = query_engine.query("What are the key points discussed in the meeting?")
print("Response from the query engine:")
print(response)